In [ ]:
# ruff: noqa: T201, T203, ERA001, BLE001
from __future__ import annotations

import os
from pathlib import Path

while Path.cwd().name == "notebooks" or Path.cwd().name == "inference":
    os.chdir("..")

import json
import logging
from itertools import product

import pandas as pd
from dotenv import load_dotenv

import src.prompts.system as system_prompts
from config.config import DATA_DIR
from config.runs_vars import SUBSET_TO_DATASET_MAP
from src.evaluate import (
    get_predictions_with_gt,
    plot_usage_histograms,
    save_analysis_results,
    store_run_metrics_df,
)
from src.formatting import (
    format_predictions_run_path,
    format_storing_pathes_from_run_path,
)
from src.LLM_servers.openai import OpenAIServer
from src.processing import parallel_samples_process, try_load_partial_results, try_load_precomputed_oracle_pairs
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)
from src.utils import expand_setups, save_run_results

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

### Run experiments


In [ ]:
MAX_WORKERS = 1500
MODELS = [
    # "gemini-2.5-flash-lite",
    "gemini-2.0-flash-lite",
    # "gemini-2.5-pro"
    # "gemini-2.5-flash-lite",
    # "gpt-4.1-nano"
]  # "gpt-4o-mini", "gemini-2.5-flash-preview-04-17", "gemini-2.0-flash-lite", "gemini-1.5-flash", "gemini-2.0-flash"

DATASETS_MAP = {
    # "bioml-2024": ["omim-ordo", "snomed-ncit.pharm", "ncit-doid", "snomed-fma.body"][:0],  # "snomed-ncit.neoplas"],  # "snomed-fma.body"]],
    "largebio": ["fma-nci", "fma-snomed"][1:],  # "snomed-nci"],  # , ],
    # "anatomy": ["human-mouse"],
}
SUFFIX = "_reduced"
# from src.constants import BinaryOutputFormat, BinaryOutputFormatWithReasoning

# EXP_NAMES_MAP = {"absent": {}, "v1": few_shots}
# EXP_NAMES_MAP = {
#     "no-reasoning": BinaryOutputFormat,
#     "with-reasoning": BinaryOutputFormatWithReasoning,
# }
EXPERIMENT_TYPE = "n_few_shots"  # "few-shot", "reasoning"
EXP_NAMES_MAP = {
    f"v{version}/{n}": n
    for n in (0, 1, 2, 3, 5, 8, 8, 11, 15, 20, 26, 33, 41, 50, 62, 75)[::-1]
    for version in range(5, 10)
}

onto_src, onto_tgt = None, None

PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        prompt_direct_entity,
        # prompt_direct_entity_ontological,
        prompt_direct_entity_with_synonyms,
        # prompt_sequential_hierarchy,
        # prompt_sequential_hierarchy_ontological,
        # prompt_sequential_hierarchy_with_synonyms,
    ]
}

SETUPS = []
for DATASET, SUBSETS in DATASETS_MAP.items():
    for SUBSET, EXP_NAME, MODEL in product(SUBSETS, EXP_NAMES_MAP.keys(), MODELS):
        setup = {
            "subset": SUBSET,
            "model": MODEL,
            "experiment_name": EXP_NAME,
            "max_workers": MAX_WORKERS,
            "suffix": SUFFIX,
            "prompt_functions": list(PROMPT_FUNCTIONS_MAP.keys()),
            "experiment_type": EXPERIMENT_TYPE,
        }
        SETUPS.append(setup)


In [ ]:
BASE_SETUP = {
    "subset": "human-mouse",
    "model": "gemini-2.0-flash-lite",
    "experiment_name": "v0/0",
    "max_workers": 400,
    "suffix": "_reduced",
    "experiment_type": "n_few_shots",
    "prompt_functions": ["direct_entity", "direct_entity_with_synonyms"],
}

SETUPS = [
    {
        "subset": ["fma-nci", "fma-snomed"],
        "model": ["gemini-2.0-flash-lite", "gpt-4.1-nano"],
        "experiment_name": [f"v{v}/{n}" for v in range(5, 7) for n in (1, 3)],
        "max_workers": [100, 500],
        "suffix": "_reduced",
        "experiment_type": "n_few_shots",
        "prompt_functions": [["direct_entity", "direct_entity_with_synonyms"]],
    }
]

SETUPS = [
    {
        "subset": "omim-ordo",
        "experiment_name": "v0/15",
        "prompt_functions": ["direct_entity_with_synonyms"],
        "model": "gemini-2.5-flash-lite",
    },
]
SETUPS = expand_setups(SETUPS, BASE_SETUP, exclude_keys=["prompt_functions"])

len(SETUPS)

In [ ]:
for setup in SETUPS:
    SUBSET = setup["subset"]
    DATASET = SUBSET_TO_DATASET_MAP[SUBSET]
    MODEL = setup["model"]
    EXP_NAME = setup["experiment_name"]
    MAX_WORKERS = setup["max_workers"]
    SUFFIX = setup["suffix"]
    EXPERIMENT_TYPE = setup["experiment_type"]

    PROMPT_FUNCTIONS = {name: PROMPT_FUNCTIONS_MAP[name] for name in setup["prompt_functions"]}

    llm_oracle = (
        OpenAIServer(
            api_key=os.environ["GEMINI_API_KEY"],
            base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
            max_tokens=1000,
        )
        if MODEL.startswith("gemini")
        else OpenAIServer(
            api_key=os.environ["OPENAI_API_KEY"],
            max_tokens=1000,
        )
    )
    llm_oracle.add_system_context(system_prompts.BASELINE_INITIALIZATION_MESSAGE)

    run_path = format_predictions_run_path(DATASET, SUBSET, MODEL, EXPERIMENT_TYPE, exp_spec=EXP_NAME)
    version, n_few_shots = EXP_NAME.split("/")
    n_few_shots = int(n_few_shots)

    few_shot_samples_path = DATA_DIR / f"preparation/{DATASET}/{SUBSET}/{version}.json"
    few_shots = json.load(few_shot_samples_path.open())["OpenAI"]

    print(MODEL, SUBSET, EXP_NAME)

    for prompt_name, prompt_function in PROMPT_FUNCTIONS.items():
        llm_oracle.clear_context()
        llm_oracle.add_system_context(system_prompts.INTUITIVE_NATURAL_LANGUAGE_JUDGEMENT_MESSAGE)

        for sample in few_shots[prompt_function.__name__][: 2 * n_few_shots]:
            llm_oracle.add_context(sample["input"], "user")
            llm_oracle.add_context(sample["output"], "assistant")

        prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
            run_path, SUBSET, MODEL, prompt_name, suffix=SUFFIX
        )
        oracle_candidate_pairs = try_load_precomputed_oracle_pairs(DATASET, SUBSET, prompt_function, suffix=SUFFIX)
        partial_results = try_load_partial_results(prediction_path)
        results, tokens_usage, confidences = parallel_samples_process(
            oracle_candidate_pairs,
            llm_oracle,
            onto_src=None,
            onto_tgt=None,
            model=MODEL,
            max_workers=MAX_WORKERS,
            prompt_function=prompt_function,
            partial_results=partial_results,
            sleep_time=0,
        )

        save_run_results(results, prediction_path, columns=["Source", "Target", "Prediction", "Confidence"])
        plot_usage_histograms(tokens_usage, confidences, do_plot=False, do_print=False, suptitle=prompt_name)

        try:
            predictions = get_predictions_with_gt(run_path, DATASET, SUBSET, MODEL, prompt_name, suffix=SUFFIX)
            save_analysis_results(
                predictions,
                print_results=False,
                plot_confusion_matrix=False,
                subtitle=f"{SUBSET}: {MODEL} {prompt_name} | ",
                cm_save_path=diagram_path,
                stats_path=stats_path,
            )
        except Exception as e:
            print(f"Error: {e}")

    store_run_metrics_df(PROMPT_FUNCTIONS, run_path, DATASET, SUBSET, MODEL, suffix=SUFFIX)
